# Isaacus Tabular Review Cookbook
In this cookbook, we’ll show you how to build a tabular review application using Isaacus’s legal AI stack.

Instead of relying on a hallucination-prone RAG workflow, we’ll take a more robust approach to document intelligence. We’ll turn each document into a knowledge graph, then extend it with custom extraction and entity-linking capabilities. The result is a review experience that stays closer to the source text while also offering better navigation, more precise retrieval, and a more interactive user experience than traditional RAG-based apps.

Our final app runs behind a FastAPI server and can be connected to any front end. To make things easier, we’ve included a simple client `tabular-review-client.html` that can already navigate the ILGS format and the custom column logic built throughout this cookbook.

# Step 1: Setting up our environment.

Before you begin with this guide, you will need an Isaacus account and a valid Isaacus API key. You can get one by following the first step of the [quickstart guide](https://docs.isaacus.com/quickstart).

Next, be sure to set an `ISAACUS_API_KEY` environment variable. To do this, we recommend creating a `.env` file in the same directory as this notebook with the following content:

`ISAACUS_API_KEY=insert_your_api_key_here`

Now let’s install the required dependencies, import them, and initialize the Isaacus API client.

In [ ]:
%pip install -q fastapi uvicorn isaacus qdrant-client dotenv

In [ ]:
import json
import os
import threading
import uuid
from pathlib import Path
from typing import Any

import uvicorn
from dotenv import load_dotenv
from fastapi import BackgroundTasks, Body, FastAPI, HTTPException
from fastapi.responses import FileResponse, PlainTextResponse
from isaacus import Isaacus
from isaacus.types.ilgs.v1.document import Document
from qdrant_client import QdrantClient, models

In [ ]:
load_dotenv()
isaacus_client = Isaacus(api_key=os.getenv("ISAACUS_API_KEY"))

We’ll start by setting up the core configuration for the app.

The block below defines the key constants for the client, our embedding batch size, search threshold, and app title. It also initializes the FastAPI server, creates a state for uploaded documents, and starts an Qdrant instance for vector storage and retrieval. We’ll unpack each of these pieces in more detail later.

For simplicity, everything here runs in memory. In a production deployment, you would typically replace these components with persistent storage and a more comprehensive configuration layer.

In [ ]:
# App configuration.
CLIENT_HEADERS = {"Cache-Control": "no-store"}
CLIENT_FILENAME = "tabular-review-client.html"
CLIENT_PATH = Path.cwd() / CLIENT_FILENAME 
EMBEDDING_BATCH_SIZE = 128 # Number of chunks to process in a batch for embedding and enrichment.
SIMILARITY_THRESHOLD = 0.4 # Cosine similarity threshold for vector search results. Lower = higher recall, Higher = higher precision.
APP_TITLE = "Isaacus Tabular Review Demo Server"

# App setup.
app = FastAPI(title=APP_TITLE)

APP_STATE: dict[str, Any] = {
    "docs": {},
    "doc_order": [],
}
APP_LOCK = threading.RLock()

vector_db = QdrantClient(":memory:", force_disable_check_same_thread=True)

# Step 2: Creating the base of our knowledge graph.

Normally, building something like a knowledge graph would require spending hours, if not days, designing a schema that fits a specific use case.

For some of the largest knowledge graphs in the world, like Wikidata, Wikipedia, and the legal knowledge networks behind Westlaw and LexisNexis, that process has likely taken the cumulative effort of many lifetimes.

[Kanon 2 Enricher](https://docs.isaacus.com/capabilities/enrichment) gives us a much faster way to build knowledge graphs. In a matter of milliseconds, the model can identify the key entities and relationships in a document and organize them into the [Isaacus Legal Graph Schema](https://docs.isaacus.com/ilgs/introduction), a knowledge graph schema designed to capture most of the core features present in legal documents.

That gives us a strong starting point that we can refine and extend later.

To begin, we’ll define a small wrapper function around Kanon 2 Enricher that batches input texts and returns an ILGS document for each one.

In [ ]:
def enrich(texts: list[str]) -> list[Any]:
    response = isaacus_client.enrichments.create(
        model="kanon-2-enricher",
        texts=texts,
        overflow_strategy="auto",
    )
    return [result.document for result in response.results]

Now let’s pass a small sample passage into the enricher and inspect the knowledge graph it produces.

This example is designed to highlight some of the core structures Kanon 2 Enricher can identify, including people, organizations, roles, locations, dates, and document segments. We’ll use it as a bite-sized window into the richer network of relationships and hierarchies that Kanon 2 Enricher can infer from legal text.

In [ ]:
example_text = """
    Here is a list of some of the features Kanon 2 Enricher can automatically extract and organize into a knowledge graph:
    1. Persons: John Doe wrote this document.
    2. Relationships: John Doe works at Acme Corporation as its director.
    3. Locations: Acme Corporation is located in San Francisco.
    4. Dates: This contract was signed on January 1, 2020.
    5. Segments: All of these list items will be labeled as list items in the graph.
    6. Much, much more!

    This is a confidentiality clause. This sentence will make more sense in Step 2 when we design a custom classification pipeline to label spans of the text based on a query.

    This Agreement shall be governed by the laws of the State of California. This sentence will make more sense in Step 3 when we will create a pipeline to define custom relationships between entities in the graph based on a query.
"""

ilgs_document = enrich([example_text])[0]
print(json.dumps(ilgs_document.to_dict(), indent=2))

For such a small passage, the output is already quite rich! Although the raw ILGS document is not especially easy to read, we can already see that Kanon 2 Enricher has identified multiple entity types and structured them in useful ways. In this example, it captures a natural person (`John Doe`), a corporate person (`Acme Corporation`), and a governing jurisdiction (`the State of California`), along with their roles and relationships. It also identifies two locations, classifies them by type, extracts a date labelled as a signature, and breaks the text into several document segments.

Because the raw output is span-based, it is a little hard to read as a human. To make it easier to inspect, we can decode some of those spans back into text and verify that the knowledge graph is accurate.

In [ ]:
# Temporary inspection code to verify that the ILGS knowledge graph is accurate.
# In the final application, span decoding will happen on the client side by slicing the source text using the start and end offsets attached to each entity.
person_names = [p.name.decode(ilgs_document.text) for p in ilgs_document.persons]
people = list(ilgs_document.persons)
location_names = [l.name.decode(ilgs_document.text) for l in ilgs_document.locations]
dates = list(ilgs_document.dates)
segment_kinds = [s.kind for s in ilgs_document.segments]

john = people[0]
acme = people[1]
first_date = dates[0]
first_location = location_names[0]

print(
    f"""
ILGS document summary
=====================

Persons
-------
- {person_names[0]} (id: {john.id})
- {person_names[1]} (id: {acme.id})

Relationship
------------
- {person_names[0]} is the {john.role} of {john.parent}

Locations
---------
- {", ".join(location_names)}
- {first_location} is linked via residence: {acme.residence}

Dates
-----
- {first_date.value} (type: {first_date.type})

Segments
--------
- {", ".join(segment_kinds)}
""".strip()
)

With just a small amount of code, we have already matched a large part of the review experience offered by tools like Harvey and Legora, including the extraction of parties, dates, locations, and other core legal entities. And we've done this all without calling a single generative model. For more on what Kanon 2 Enricher can classify, link, and extract, and for a deeper introduction to the ILGS schema, see the documentation [here](https://docs.isaacus.com/ilgs/).

ILGS can feel dense at first, but in practice we do not need to understand every detail of the format to build useful applications on top of it. As long as we preserve the core structure and keep track of the unique IDs produced by the enricher in our downstream classification and retrieval logic, we can always stitch the pieces back together to navigate the graph on the client side. This is precisely what we’ll be doing in the next couple of steps.

# Step 3: Extending the knowledge graph with span-level classification.
With the knowledge graph base done, we can now turn our attention to adding support for custom labelling and retrieval features, which are a core part of the AI-powered tabular review experience. To do this, we’ll use another Isaacus model: [Kanon 2 Embedder](https://docs.isaacus.com/capabilities/embedding). 

Although embedding models are commonly used for retrieval, they can also perform classification tasks by encoding spans of text and comparing them against a query framed as a label or attribute. With the right filtering logic, this gives us a flexible and reliable way to add custom span-level labels to a document and extend the knowledge graph with features tailored to a specific use case.

To get started, we’ll define two small helper functions. The first batches a list into smaller chunks. The second sends those chunks to Kanon 2 Embedder and returns vectors for either document spans or queries. We’ll build the actual classification logic on top of these helpers in the next step.

In [ ]:
# Helper function to split a list into smaller batches.
def batched(items: list[Any], size: int) -> list[list[Any]]:
    return [items[i : i + size] for i in range(0, len(items), size)]


# Embed texts for either retrieval/document or retrieval/query tasks.
def embed_texts(texts: list[str], task: str) -> list[list[float]]:
    vectors: list[list[float]] = []

    for batch in batched(texts, EMBEDDING_BATCH_SIZE):
        response = isaacus_client.embeddings.create(
            model="kanon-2-embedder",
            texts=batch,
            task=task,
        )
        vectors.extend([list(item.embedding) for item in response.embeddings])

    return vectors

If we called the function above directly, it would just return embeddings. To make those embeddings useful, we need to store them in an index that we can query later.

We also do not want to embed the full document as a single block. As with RAG, retrieval, and classification work best over meaningful spans of text. Those spans might be sentences, clauses, phrases, or larger multi-sentence sections, depending on the task.

Choosing the level of span granularity by hand can be difficult. One advantage of Kanon 2 Enricher is that it already breaks the document into meaningful, hierarchical segments. That gives us a natural set of embedding units while preserving the structure of the original document.

To store those embeddings, we’ll use Qdrant, a vector database that supports vector search with custom metadata. We can use the unique IDs from the ILGS document as metadata on each vector, which makes it easy to link every result back to the original span in the knowledge graph.

The code below sets up the document record, collects the ILGS segments we want to embed, and builds a Qdrant index for each document. That gives us a clean foundation for span-level retrieval and classification later on.

In [ ]:
# Create a new document record in app state.
def create_record(file_name: str, ilgs_document: Document) -> dict[str, Any]:
    file_id = str(uuid.uuid4())
    record = {
        "file_id": file_id,
        "file_name": file_name,
        "ilgs_document": ilgs_document,
        "columns": {},
        "index_status": "pending",
    }

    with APP_LOCK:
        APP_STATE["docs"][file_id] = record
        APP_STATE["doc_order"].append(file_id)

    return record


# Retrieve a stored document record or raise a 404 if it does not exist.
def get_record(file_id: str) -> dict[str, Any]:
    with APP_LOCK:
        record = APP_STATE["docs"].get(file_id)

    if record is None:
        raise HTTPException(status_code=404, detail=f"Document not found: {file_id}")

    return record


# Collect all non-empty segment spans from a document.
def collect_segments(doc: Document) -> list[tuple[str, str]]:
    segments: list[tuple[str, str]] = []

    for segment in doc.segments:
        text = segment.span.decode(doc.text).strip()
        if text:
            segments.append((segment.id, text))

    return segments


# Build a Qdrant index for a document using ILGS segments as embedding spans.
def build_index(file_id: str) -> None:
    record = get_record(file_id)
    doc = record["ilgs_document"]
    segments = collect_segments(doc)

    if not segments:
        return

    texts = [text for _, text in segments]
    vectors = embed_texts(texts, task="retrieval/document")
    collection_name = f"doc_{file_id.replace('-', '_')}"

    points = [
        models.PointStruct(
            id=i,
            vector=vector,
            payload={
                "segment_id": segment_id,
                "text": text,
            },
        )
        for i, ((segment_id, text), vector) in enumerate(zip(segments, vectors))
    ]

    with APP_LOCK:
        if vector_db.collection_exists(collection_name):
            vector_db.delete_collection(collection_name)

        vector_db.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(
                size=len(vectors[0]),
                distance=models.Distance.COSINE,
            ),
        )

        vector_db.upsert(collection_name=collection_name, points=points)

        record["collection_name"] = collection_name
        record["index_status"] = "ready"

With the corpus logic in place, we can now turn to querying the index for retrieval and classification.

We start with two small helper functions: `spans_overlap`, which checks whether two ILGS spans overlap, and `segment_lookup`, which maps segment IDs back to their corresponding segment objects. These will make it easier to work with span-based results later on.

We then define `ensure_index`, which makes sure a document’s vector index has been built and updates the record with progress flags along the way. This is important because our code is designed to sit inside a live asynchronous application, where we will need to keep track of whether a document is still being processed or is ready to query.

Finally, we define `delete_record`, which removes a document from app state and cleans up its associated Qdrant collection when the document is deleted.

In [ ]:
# Return True if two ILGS spans overlap.
def spans_overlap(a: Any, b: Any) -> bool:
    return bool(a and b) and a.start < b.end and a.end > b.start


# Map segment IDs back to segment objects.
def segment_lookup(doc: Document) -> dict[str, Any]:
    return {segment.id: segment for segment in doc.segments}


# Ensure the vector index for a document exists, building it if needed.
def ensure_index(file_id: str) -> dict[str, Any]:
    record = get_record(file_id)
    if record["index_status"] == "ready":
        return record

    with APP_LOCK:
        record["index_status"] = "building"

    build_index(file_id)

    with APP_LOCK:
        record["index_status"] = "ready"

    return record


# Delete a document record and its associated Qdrant collection.
def delete_record(file_id: str) -> dict[str, Any]:
    record = get_record(file_id)
    collection_name = f"doc_{file_id.replace('-', '_')}"

    with APP_LOCK:
        if vector_db.collection_exists(collection_name):
            vector_db.delete_collection(collection_name)

        APP_STATE["docs"].pop(file_id, None)
        APP_STATE["doc_order"] = [
            item for item in APP_STATE["doc_order"] if item != file_id
        ]

    return record

Now we can define the main querying logic for span-level retrieval.

In a standard semantic search pipeline, we would query the vector database and return the most relevant segments. Here, though, we want to treat the query more like a classification prompt, so we also need a way to filter out segments that are not similar enough to be meaningful matches.

To do that, we apply a similarity threshold to the scores returned by Qdrant and keep only the segments above that cutoff. Kanon 2 Embedder was not explicitly trained around a fixed classification threshold, but in practice a cosine similarity of `SIMILARITY_THRESHOLD = 0.4` works well as a starting point. The right threshold will still depend on the task and the kind of labels you are trying to surface.

The code below applies that threshold, resolves the matching vectors back to ILGS spans, and then deduplicates overlapping results. If both a parent span and one of its children are returned, we keep only the parent span, since it usually carries more useful context for downstream review.

In [ ]:
# Run semantic search over Qdrant hits, then resolve them back to ILGS spans.
def search_spans(
    file_id: str,
    query: str,
    threshold: float = SIMILARITY_THRESHOLD,
) -> list[dict[str, Any]]:
    record = ensure_index(file_id)
    collection_name = f"doc_{file_id.replace('-', '_')}"

    if not vector_db.collection_exists(collection_name):
        return []

    query_vector = embed_texts([query], "retrieval/query")[0]

    with APP_LOCK:
        response = vector_db.query_points(
            collection_name=collection_name,
            query=query_vector,
            score_threshold=threshold,
        )

    doc = record["ilgs_document"]
    segments = segment_lookup(doc)

    hits: list[tuple[Any, float, str | None]] = []
    for point in response.points:
        payload = point.payload or {}
        segment_id = payload.get("segment_id")
        if not segment_id or segment_id not in segments:
            continue

        hits.append(
            (
                segments[segment_id],
                float(point.score),
                payload.get("text"),
            )
        )

    # Prefer larger spans first so parent spans win over overlapping children.
    hits.sort(
        key=lambda item: (
            -(item[0].span.end - item[0].span.start),
            -item[1],
        )
    )

    deduped_hits: list[tuple[Any, float, str | None]] = []
    for segment, score, text in hits:
        if any(spans_overlap(segment.span, existing[0].span) for existing in deduped_hits):
            continue
        deduped_hits.append((segment, score, text))

    # Return final results ordered by score.
    deduped_hits.sort(key=lambda item: item[1], reverse=True)

    return [
        {
            "score": score,
            "text": text or segment.span.decode(doc.text).strip(),
            "metadata": {
                "segment_id": segment.id,
                "start": segment.span.start,
                "end": segment.span.end,
            },
        }
        for segment, score, text in deduped_hits
    ]

With the helper functions in place, we can now try out span-level retrieval and classification on our example text.

In this case, we’ll look for segments related to confidentiality obligations. We can phrase that as a natural-language query, embed it as a retrieval vector, and use our span search logic to return the most relevant segments from the document’s Qdrant index.

The code below creates a document record, builds its vector index, runs the query, and prints the matching spans along with their scores and character offsets.

In [ ]:
example_record = create_record(
    file_name="example.txt",
    ilgs_document=ilgs_document,
)
build_index(example_record["file_id"])
example_record["index_status"] = "ready"

query = "What are the confidentiality obligations outlined in this agreement?"
results = search_spans(example_record["file_id"], query)

for result in results:
    print(f"Score: {result['score']:.3f}")
    print(f"Character offsets: {result['metadata']['start']}:{result['metadata']['end']}")
    print(result["text"])
    print()

Sure enough, our classification logic has surfaced the passage that best answers our query! With traditional chunking and retrieval techniques, we would not be able to classify a span at this level of granularity. Indeed, in many legal AI applications the same query would return an entire chunk, likely with the irrelevant governing law clause included in the context. When context is contaminated like this, it not only makes for a worse reviewing experience, but it also poisons the context window of generative models in RAG pipelines, leading to a higher incidence of hallucinations.

We’ll delete this test record so it does not show up in the app later. Before doing that, though, it is useful to inspect the record format directly, since this is the same structure the server will keep in memory and eventually send back to the client.

To keep the output readable, we’ll remove the full ILGS document before printing the record.

In [ ]:
deleted_record = delete_record(example_record["file_id"])

record_preview = {
    **deleted_record,
    "ilgs_document": "ILGS document data removed for brevity.",
}

print(
    "Here is what a stored record looks like:\n"
    + json.dumps(record_preview, indent=2, default=str)
)

## Step 4: Extending the knowledge graph with custom entity linking and relationship extraction

At this point, the app can already reproduce much of the review experience offered by tools like Harvey and Legora, and then some, all without relying on a generative model. But one of the main advantages of working with a knowledge graph is that we are not limited to the relationships extracted by the base model, or to whatever noisy result a generative model might produce. We can define new, custom relationships on top of the graph at query time based on the user’s needs.

To do that, we’ll use [Kanon Answer Extractor](https://docs.isaacus.com/capabilities/extractive-question-answering). Given a user query, it returns answer spans from the source document. We can then cross-reference those spans against the ILGS document and check which entities are mentioned within them. Any matching entities can be linked back to the query, effectively creating new relationships on the fly without having to define them in advance.

The code below does exactly that. It checks whether an entity mention overlaps with an extracted answer span, then collects the IDs of the matching entities so they can be linked back to the original ILGS document later.

Note that our QA extraction endpoint uses an older base model, so its performance may vary. We are currently developing a newer universal extraction endpoint. For an alternative model, consider using GLiNER2.

In [ ]:
# If an entity mention overlaps with an extracted answer span, save its ID so
# it can be linked back to the original ILGS document later.
def link_entities(
    entities: list[Any],
    answer: Any,
    entity_ids: list[str],
    seen: set[str],
) -> None:
    for entity in entities:
        spans = list(entity.mentions)
        if any(spans_overlap(span, answer) for span in spans) and entity.id not in seen:
            seen.add(entity.id)
            entity_ids.append(entity.id)


# Run QA extraction for a query, then link any overlapping ILGS entities back
# to the extracted answer spans.
def extract_entities(doc: Document, query: str) -> list[str]:
    response = isaacus_client.extractions.qa.create(
        model="kanon-answer-extractor",
        query=query,
        texts=[doc.text],
        ignore_inextractability=False,
    )

    entity_ids: list[str] = []
    seen: set[str] = set()

    for answer in response.extractions[0].answers:
        link_entities(doc.persons, answer, entity_ids, seen)
        link_entities(doc.locations, answer, entity_ids, seen)
        link_entities(doc.terms, answer, entity_ids, seen)

    return entity_ids

Let’s now try this code out on our example document.

The extraction pipeline should identify the span that best answers the query, then link any entities mentioned in that span back to the original ILGS document. In this example, we want to identify the governing law jurisdiction of the agreement. Under the ILGS schema, that answer can surface through more than one part of the graph. Here, the State of California appears as both a political entity and a linked location. Because they are already linked via the residence attribute, this single query is actually performing multiple links at once, highlighting the power of approaching document review as a knowledge graph problem.

In [ ]:
extracted_entity_ids = extract_entities(
    ilgs_document,
    "What is the governing law jurisdiction of this agreement?",
)

entity_lookup = {
    entity.id: entity
    for entity in (
        ilgs_document.persons + ilgs_document.locations + ilgs_document.terms
    )
}

entity = entity_lookup[extracted_entity_ids[0]]
entity_location = entity_lookup[extracted_entity_ids[1]]
entity_location_name = entity_location.name.decode(ilgs_document.text)
entity_location_type = entity_location.type
entity_name = entity.name.decode(ilgs_document.text) 

print("Linked governing law entity")
print("---------------------------")
print(f"Name: {entity_name}")
print(f"Type: {entity.type}")
print(f"ID:   {entity.id}")
print(f"Linked location: {entity_location_name} ({entity_location_type.title()})")

We've now linked the governing law jurisdiction to the entity of the State of California, which is itself linked to the location of California. If we wanted to extend this entity with more labels and relationships, all we'd need to do is send more queries and the network would automatically expand as everything is linked using the unique ids first extracted by the enrichment model.

To create a single endpoint that the server can use, we will wrap the custom querying logic into a simple helper. Feel free to extend this function with your own custom extractive, linking or question-answering features. For now, we'll go with two modes of querying: span-level retrieval for classification-style columns, and entity extraction for relationship-style columns.

In [ ]:
# Run the appropriate classification or entity linking pipeline based on the column type supplied.
def classify_document(file_id: str, query: str, col_type: str) -> dict[str, Any]:
    record = ensure_index(file_id)

    if col_type == "span":
        return {"col_data": search_spans(file_id, query)}

    if col_type == "entity":
        return {"col_data": extract_entities(record["ilgs_document"], query)}

    raise ValueError(f"Unsupported column type: {col_type}")

## Step 5: Building the server.
With entity linking complete, we now have all the core intelligence capabilities we need to power the tabular review app. 

To recap, we have:
1. Created a knowledge graph from each document using Kanon 2 Enricher.
2. Extended that graph with custom span-level classification using Kanon 2 Embedder and Qdrant.
3. Added custom entity linking and relationship extraction using the Isaacus QA extraction endpoint.

All that remains is to expose this logic through a live server that can receive requests from the client, run the appropriate enrichment and querying logic, and return the results in a format the UI can display.

To get started, we’ll define two small helpers. The first packages a stored document record into the response shape expected by the client. The second triggers index building in the background when a document is uploaded, so the document is ready to query by the time the user interacts with it.

In [ ]:
# Convert a stored record into the response shape expected by the client.
def package_record(record: dict[str, Any]) -> dict[str, Any]:
    return {
        "doc_id": record["file_id"],
        "file_name": record["file_name"],
        "document": record["ilgs_document"].model_dump(mode="json"),
        "columns": record["columns"],
        "index_status": record["index_status"],
    }

# Trigger index building for a document if it is not already ready.
def build_index_in_background(file_id: str) -> None:
    if APP_STATE["docs"][file_id]["index_status"] != "ready":
        ensure_index(file_id)


Now let’s define the server logic for handling the initial request that turns uploaded documents into ILGS. To keep this cookbook simple, we’ve pushed document conversion onto the client. That means the server expects a request body in the following shape:

```python
{
    "documents": [
        {
            "file_name": str,
            "text": str,
        },
        ...
    ]
}
```
So, for each document, the server will:

1. Convert the raw text into ILGS using Kanon 2 Enricher.
2. Create a document record in app state.
3. Trigger embedding and indexing in the background using Kanon 2 Embedder and Qdrant.
4. Return the packaged record to the client immediately.

It is important to separate enrichment from indexing. We want the client to render the document as soon as enrichment is complete, while indexing continues in the background.

The response shape looks like this, with column data added in later queries:
```python
{
    "doc_id": "uuid-string",
    "file_name": "contract_1.txt",
    "document": { "...full ILGS document json..." },
    "columns": {},
    "index_status": "pending"
}
```

In [ ]:
# Serve the client HTML file at the root endpoint.
@app.get("/")
def index():
    return FileResponse(str(CLIENT_PATH), headers=CLIENT_HEADERS)


# Receive documents from the client, convert them to ILGS, create records,
# trigger background indexing, and return the packaged records to the UI.
@app.post("/review")
def review(body: dict[str, Any], background_tasks: BackgroundTasks):
    lines: list[str] = []

    for batch in batched(body["documents"], EMBEDDING_BATCH_SIZE):
        texts = [str(doc["text"]) for doc in batch]
        ilgs_documents = enrich(texts)

        for input_doc, ilgs_document in zip(batch, ilgs_documents):
            record = create_record(
                file_name=str(input_doc["file_name"]),
                ilgs_document=ilgs_document,
            )
            lines.append(json.dumps(package_record(record), ensure_ascii=False))
            background_tasks.add_task(build_index_in_background, record["file_id"])

    return PlainTextResponse(
        "\n".join(lines),
        media_type="application/x-ndjson",
        headers={**CLIENT_HEADERS, "X-Tabular-Review-Version": APP_TITLE},
    )

We then define the endpoints for custom column queries and document deletion.

For column queries, the server receives a list of document IDs, a column type, and a natural-language query. It then runs the appropriate retrieval or extraction logic for each document and returns the results to the client for display in the UI.

For document deletion, the server removes the document from app state and deletes its corresponding vector index from Qdrant to keep everything clean and in sync between the server and the client.

In [ ]:
# Receive a custom column query, run the appropriate retrieval or extraction
# logic for each document, and return the results to the client.
@app.post("/column-query")
def column_query(body: dict[str, Any] = Body(default_factory=dict)) -> dict[str, Any]:
    doc_ids = list(body["doc_ids"])
    query = str(body["query"]).strip()
    col_type = str(body["col_type"]).strip()

    column_id = str(uuid.uuid4())
    results: list[dict[str, Any]] = []

    for file_id in doc_ids:
        col_data = classify_document(file_id, query, col_type)["col_data"]

        get_record(file_id)["columns"][column_id] = {
            "column_id": column_id,
            "query": query,
            "col_type": col_type,
            "col_data": col_data,
        }

        results.append({"doc_id": file_id, "col_data": col_data})

    if len(results) == 1:
        return {"column_id": column_id, "col_data": results[0]["col_data"]}

    return {"column_id": column_id, "results": results}


# Delete a document and its associated vector index.
@app.delete("/docs/{doc_id}")
def delete_doc(doc_id: str) -> dict[str, Any]:
    record = delete_record(doc_id)
    return {
        "deleted": True,
        "doc_id": doc_id,
        "file_name": record["file_name"],
    }

## Step 6: Going live

We’ve now reached the final step of the cookbook. At this point, we have everything we need to run the tabular review app: a pipeline for building and extending the knowledge graph, and a server that can process client requests and return results in a format the UI can use.

All that remains is to start the server and connect it to the client. To save time, we’ve included a front end that can navigate the ILGS format and work with the custom column logic we built throughout this cookbook.

Our frontend can do things tools like Harvey and Legora cannot. Rather than treating review as a sequence of extracted answers inside a table, it turns the document into a navigable legal knowledge graph. This empowers users to inspect the source text, move through linked entities and sections, and verify how every result connects back to the document itself—all in a Wikipedia-like interface.

That experience is only possible because of the structured knowledge graph we built with Kanon 2 Enricher and extended with custom logic powered by the Isaacus stack. By keeping the intelligence grounded in the original text and linking every result back to its source, we can create a review experience that is more interactive, fluent, and easier to trust, while still supporting powerful retrieval and classification workflows.

To use the app, simply run the server code in this notebook, then open the app at your chosen address. From there, you can upload files, add new columns with natural-language queries, and explore the resulting knowledge graph through both an entity previewer and a document viewer that annotates the original text using the ILGS format.

We encourage you to build on top of this foundation and adapt the experience to fit your own workflow and use case.

In [ ]:
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run_server, daemon=True).start()
print("Server started at http://127.0.0.1:8000")